[DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash) (C) Fabrizio Smeraldi, 2020,2024 ([f.smeraldi@qmul.ac.uk](mailto:f.smeraldi@qmul.ac.uk) - [web](http://www.eecs.qmul.ac.uk/~fabri/)). This notebook is released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/).

# COVID-19 Dashboard: Hospital Admissions & Deaths (Rolling Mean)

This dashboard displays the recorded data obtained via the UK Health Security Agency (UKHSA) COVID-19 dashboard API with the aims to dig deeper into the logged hospital admissions and death rated during COVID-19 in England.

There are 2 mean metrics illustrated in these graphs:
- **COVID-19_healthcare_admissionRollingMean** - shows the daily number of hospital admissions for those patients with diagnosed COVID-19
- **COVID-19_deaths_ONSRollingMean** - displays the mortality rates for COVID-19 based upon ONS (Office of National Statistics) registrations.

This dashboard:
- Utilises **offline JSON files** so that when the data is called it can work whilst connected to the internet and also if access to the internet is not available.
- Next, it illustrates **two interactive graphs**, which can be each filtered out to show Admissions, Deaths or Both using widgets.
- Lastly, it provides a **Refresh data** button that has the ability to call the API as well as updating both the graphs.

In [1]:
import requests
import time
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import clear_output
import ipywidgets as wdg
%matplotlib inline
# make figures larger
plt.rcParams['figure.dpi'] = 100

In [2]:
class APIwrapper:
    #Attributes
    _access_point="https://api.ukhsa-dashboard.data.gov.uk"
    _last_access=0.0 
    #Methods
    def __init__(self,theme, sub_theme, topic, geography_type, geography, metric):
        url_path=(f"/themes/{theme}/sub_themes/{sub_theme}/topics/{topic}/geography_types/" +
                  f"{geography_type}/geographies/{geography}/metrics/{metric}")
        self._start_url=APIwrapper._access_point+url_path
        self._filters=None
        self._page_size=-1
        self.count=None

    def get_page(self, filters={}, page_size=5):
        # Check page size is within range
        if page_size>365:
            raise ValueError("Max supported page size is 365")
        # restart from first page if page or filters have changed
        if filters!=self._filters or page_size!=self._page_size:
            self._filters=filters
            self._page_size=page_size
            self._next_url=self._start_url
        # signal the end of data condition
        if self._next_url==None: 
            return [] # we already fetched the last page
        # simple rate limiting to avoid bans
        curr_time=time.time() # Unix time: number of seconds since the Epoch
        deltat=curr_time-APIwrapper._last_access
        if deltat<0.33: # max 3 requests/second
            time.sleep(0.33-deltat)
        APIwrapper._last_access=curr_time
        # build parameter dictionary by removing all the None
        # values from filters and adding page_size
        parameters={x: y for x, y in filters.items() if y!=None}
        parameters['page_size']=page_size
        # the page parameter is already included in _next_url.
        # This is the API access. Response is a dictionary with various keys.
        # the .json() method decodes the response into Python object (dictionaries,
        # lists; 'null' values are translated as None).
        response = requests.get(self._next_url, params=parameters).json()
        # update url so we'll fetch the next page
        self._next_url=response['next']
        self.count=response['count']
        # data are in the nested 'results' list
        return response['results'] 

    def get_all_pages(self, filters={}, page_size=365):
        data=[] # build up all data here
        while True:
            # use get_page to do the job, including the pacing
            next_page=self.get_page(filters, page_size)
            if next_page==[]:
                break # we are done
            data.extend(next_page)
        return data

In [3]:
#Contacting API and building Json data:
#structure = {
#    "theme": "infectious_disease", 
#    "sub_theme": "respiratory",
#    "topic": "COVID-19",
#    "geography_type": "Nation", 
#    "geography": "England"
#}

In [4]:
#Downloading data for admissions metric:
#structure ["metric"] = "COVID-19_healthcare_admissionRollingMean"
#api = APIwrapper(**structure)
#admissions_data = api.get_all_pages()

In [5]:
#structure["metric"] = "COVID-19_deaths_ONSRollingMean"
#api = APIwrapper(**structure)
#deaths_data = api.get_all_pages()

In [6]:
# Saving metrics as JSON files:
#with open ("admissions.json", "wt") as f:
#    json.dump(admissions_data, f)
#with open ("deaths.json", "wt") as f:
#    json.dump(deaths_data, f)

In [7]:
#Loading JSON files:
#with open("admissions.json", "rt") as INFILE:
#    admissions_data=json.load(INFILE)
#with open("deaths.json", "rt") as INFILE:
#    deaths_data=json.load(INFILE)

# Data source & loading:
Firstly, this dashboard loads the downloaded JSoN files:
- 'admissions.json' - rolling mean hospital admission data.
- 'deaths.json' - rolling mean death data.

These files were initally called from the UKHSA dashboard API via these parameters:

-    "theme": "infectious_disease", 
-    "sub_theme": "respiratory",
-    "topic": "COVID-19",
-    "geography_type": "Nation", 
-    "geography": "England"
-    "metric_name" = "COVID-19_healthcare_admissionRollingMean" or "COVID-19_deaths_ONSRollingMean"

This allows the dashboard to still run effectively even if there is no internet connection available e.g, if API is unexpectedly unavailable there is a back up. 



In [8]:
# Load saved JSON files:
import json

with open("admissions.json", "rt") as infile:
    admissions_data = json.load(infile)

with open("deaths.json", "rt") as infile:
    deaths_data = json.load(infile)
    
jsondata={
    "admissions": admissions_data,
    "deaths": deaths_data
}  
jsondata

{'admissions': [{'theme': 'infectious_disease',
   'sub_theme': 'respiratory',
   'topic': 'COVID-19',
   'geography_type': 'Nation',
   'geography': 'England',
   'geography_code': 'E92000001',
   'metric': 'COVID-19_healthcare_admissionRollingMean',
   'metric_group': 'healthcare',
   'stratum': 'default',
   'sex': 'all',
   'age': 'all',
   'year': 2020,
   'month': 8,
   'epiweek': 32,
   'date': '2020-08-07',
   'metric_value': 58.71,
   'in_reporting_delay_period': False},
  {'theme': 'infectious_disease',
   'sub_theme': 'respiratory',
   'topic': 'COVID-19',
   'geography_type': 'Nation',
   'geography': 'England',
   'geography_code': 'E92000001',
   'metric': 'COVID-19_healthcare_admissionRollingMean',
   'metric_group': 'healthcare',
   'stratum': 'default',
   'sex': 'all',
   'age': 'all',
   'year': 2020,
   'month': 8,
   'epiweek': 32,
   'date': '2020-08-08',
   'metric_value': 62.71,
   'in_reporting_delay_period': False},
  {'theme': 'infectious_disease',
   'sub_th

## Wrangle data

This section converts the JSON structure provided by the API to more clean, suitable 'pandas' DataFrame format, so that it can be plotted into graphs efficiently.

1. **Combine metrics**
Firstly, the admissions and deaths data is logged into one clean dictionary - that is keyed by date.

2. **Time-series DataFrame Built**
Then, the 'timeseriesdf' DataFrame is generated with 2 main components:
- a 'DatetimeIndex' (daily dates)
- 'Admissions' & 'Deaths' columns

3. **Error Handling - Missing Values**
- Data can have null entries for certain metrics as they have different date time entries, so to ensure no errors occur in this case, the missing metric is filled with '0.0'.


In [9]:
def wrangle_data(rawdata):
    data={}
#Admissions:
    for entry in rawdata['admissions']:
        date = entry['date']
        value = entry['metric_value']

    #create an empty dictionary will be keyed by date:
        if date not in data:
            data[date]={}

        #store the admissions value under this date:
        data[date]['Admissions'] = value
#Deaths:
    for entry in rawdata['deaths']:
        date = entry['date']
        value = entry['metric_value']

        if date not in data:
           data[date] = {}

        data[date]['Deaths'] = value
#Turning dictionary into DataFrame:
    df = pd.DataFrame.from_dict(data, orient='index')
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)
#If some dates have only 1 metric, fill missing cells wih 0
    df.fillna(0.0, inplace=True)
    return df

In [10]:
data={}
for dataset in [admissions_data, deaths_data]:
    for entry in dataset:
        date=entry['date']
        metric=entry['metric']
        value=entry['metric_value']
        if date not in data:
            data[date]={}
        data[date][metric]=value

dates=list(data.keys())
dates.sort()
date

'2024-01-12'

In [11]:
def parse_date(datestring):
    return pd.to_datetime(datestring, format="%Y-%m-%d")

In [12]:
startdate=parse_date(dates[0])
enddate=parse_date(dates[-1])
print (startdate, ' to ', enddate)

2020-02-05 00:00:00  to  2025-10-31 00:00:00


In [13]:
index=pd.date_range(startdate, enddate, freq='D')
timeseriesdf=pd.DataFrame(index=index, columns=['admissions', 'deaths'])
timeseriesdf

,admissions,deaths
2020-02-05,NaN,NaN
2020-02-06,NaN,NaN
2020-02-07,NaN,NaN
2020-02-08,NaN,NaN
2020-02-09,NaN,NaN
...,...,...
2025-10-27,NaN,NaN
2025-10-28,NaN,NaN
2025-10-29,NaN,NaN
2025-10-30,NaN,NaN


In [14]:
metrics ={'admissions': 'COVID-19_healthcare_admissionRollingMean',
          'deaths': 'COVID-19_deaths_ONSRollingMean'}

for date, entry in data.items(): # each entry is a dictionary with cases, admissions and deaths
    pd_date=parse_date(date) # convert to Pandas format
    for column in ['admissions', 'deaths']: 
        metric_name=metrics[column]
        # do not assume all values are there for every date - if a value is not available, insert a 0.0
        value= entry.get(metric_name, 0.0)
        # this is the way you access a specific location in the dataframe - use .loc
        # and put index,column in a single set of [ ]
        timeseriesdf.loc[date, column]=value
            
# fill in any remaining "holes" due to missing dates
timeseriesdf.fillna(0.0, inplace=True)           
timeseriesdf

,admissions,deaths
2020-02-05,0.0,0.29
2020-02-06,0.0,0.14
2020-02-07,0.0,0.14
2020-02-08,0.0,0.14
2020-02-09,0.0,0.0
...,...,...
2025-10-27,197.14,0.0
2025-10-28,189.86,0.0
2025-10-29,180.86,0.0
2025-10-30,173.43,0.0


In [15]:
from IPython.display import clear_output
import ipywidgets as wdg
import pandas as pd
import matplotlib.pyplot as plt

In [16]:
%matplotlib inline
# make figures larger
plt.rcParams['figure.dpi'] = 100

## Graph 1: Daily COVID- 19 hospital admissions and deaths over time (rolling mean)

First Graph:
Is a **time-series plot** illustrating the hospital deaths and admissions related to COVID-19 have been changing throughout the years of 2020-2025 in England, UK. (rolling-mean values are used to avoid any anomalies)

- If **Both** option is chosen, the graph will display both the rolling-mean hospital admissions and deaths
- If **Admissions only** or **Deaths only** is chosen, the graph will only display that single metric selected.

In [17]:
def plot_timeseries(choice):
    """ Time-series graph: showing admissions and deaths (Rolling Mean) """
    if choice == 'Admissions only':
        ax = timeseriesdf [['admissions']].plot(figsize=(10,5))
    elif choice == 'Deaths only':
        ax = timeseriesdf[['deaths']].plot(figsize=(10,5))
    else: #show both
        ax = timeseriesdf[['admissions', 'deaths']].plot(figsize=(10,5))

    ax.set_title("Daily COVID-19 Hospital Admissions and Deaths (Rolling Mean)")
    ax.set_ylabel("Rolling Mean Value")
    plt.show() 

# a widget
choice_dropdown=wdg.Dropdown(
    options=['Both', 'Admissions only', 'Deaths only'],
    value='Both',
    description='Show:'
)

def refresh_graph():
    """ redraw of the graph;
    this is useful when the data have been updated. """
    current=choice_dropdown.value
    if current== 'Both':
        other = 'Admissions only'
    else:
        other = 'Both'
    choice_dropdown.value = other # forces the redraw
    choice_dropdown.value = current # now we can change it back
    
# connect the plotting function and the widget    
graph1 = wdg.interactive_output(plot_timeseries, {'choice': choice_dropdown})

# actually displays the graph
display(choice_dropdown, graph1)

Dropdown(description='Show:', options=('Both', 'Admissions only', 'Deaths only'), value='Both')

Output()

## Graph 2: Distribution of COVID-19 hospital admissions vs deaths (box plot)

This graph is a **box plot** visually comparing the distributions of daily rolling-mean admissions and deaths.

It is practical as it not only clearly shows the **median**  and **overall spread** of both metrics.
It is also not solely reliant on **dates**, so it is best suitable for this sort of data as the two metrics cover different time periods.

The **widget** named **Metric** will be responsible for what data is protrayed:
If **Both** option is chosen, the graph will display both the rolling-mean hospital admissions and deaths
- If **Admissions only** or **Deaths only** is chosen, the graph will only display that single metric selected.

In [18]:
metric_radio = wdg.RadioButtons(
    options=['Both', 'Admissions only', 'Deaths only'],
    value = 'Both', # Defaults to 'Both'
    description='Metric:',
    disabled=False
)

def plot_box_widget(choice):
    # Create a new figure and axes for the boxplot 
    fig, ax = plt.subplots(figsize=(10, 5))

    if choice == 'Both':
        choice = ['admissions', 'deaths']
    elif choice == 'Admissions only':
        choice = ['admissions']
    else: #If only want to view deaths
        choice = ['deaths']
         
    timeseriesdf[choice].astype(float).boxplot(ax=ax)
    ax.set_title("Distribution Comparison: Admissions vs Deaths (Rolling Mean)")
    ax.set_ylabel("Rolling Mean Value")
    plt.show()
#Connect plot function with widget function
graph2 = wdg.interactive_output(
    plot_box_widget,
    {'choice': metric_radio}
)
#Displays the graph:
display(metric_radio, graph2)

RadioButtons(description='Metric:', options=('Both', 'Admissions only', 'Deaths only'), value='Both')

Output()

## Refreshing data from the UKHSA API

This **Refresh data** button will go collect the most current logged data from the UKHSA dashboard API.

If the button is selected:
- The access_api() function will call the API in search for both metric values.
- These new JSON data collected will be automatically implemented into the dictionary with the same structure as the offline data saved previously.
- Then the wrangle_data() function will also be called once more to generate the timeseriesdf DataFrame again incorporating the new,updated JSON data.
- Lastly, the referesh_graph() function will also update the widgets, so that the graphs illustarte the new data obtained.

In [19]:
def access_api():
    theme = "infectious_disease"
    sub_theme = "respiratory"
    topic = "COVID-19"
    geo_type = "Nation"
    geo_name = "England"
    
#Admissions Request: 
    api_adm = APIwrapper(
        theme, sub_theme, topic, geo_type, geo_name, "COVID-19_healthcare_admissionRollingMean"
    )
    admissions_raw = api_adm.get_all_pages()

 #Deaths Request:
    api_death = APIwrapper(
        theme,sub_theme,topic,geo_type,geo_name, 
        "COVID-19_healthcare_admissionRollingMean"
    )
    deaths_raw = api_death.get_all_pages()

    apidata = {
        "admissions": admissions_raw,
        "deaths": deaths_raw,
    }
    return apidata # return data read from the API

In [20]:
def api_button_callback(button):
    apidata=access_api()
    # wrangle the data and overwrite the dataframe for plotting
    global jsondata, timeseriesdf
    jsondata = apidata
    timeseriesdf=wrangle_data(jsondata)
    refresh_graph()
    apibutton.icon="check"
    

    
apibutton=wdg.Button(
    description='Refresh data', 
    disabled=False,
    button_style='danger', 
    tooltip="Download latest UKHSA data",
    icon='download'
)

apibutton.on_click(api_button_callback) 

display(apibutton)


Button(button_style='danger', description='Refresh data', icon='download', style=ButtonStyle(), tooltip='Downl…

**Author and License** Remember that if you deploy your dashboard as a Binder it will be publicly accessible. Change the copyright notice and take credit for your work! Also acknowledge your sources and the conditions of the license by including this notice: "Based on UK Government [data](https://ukhsa-dashboard.data.gov.uk/) published by the [UK Health Security Agency](https://www.gov.uk/government/organisations/uk-health-security-agency) and on the [DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash) by Fabrizio Smeraldi. Released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/)."